# 🎯 Conditional DCGAN — Radar View Generator
### Generate Specific Radar Target Classes: Pedestrian | Electric Scooter | Car

**Difference from vanilla DCGAN:**
- Regular DCGAN: Generator receives only random noise `z` → generates *any* radar image
- **Conditional DCGAN (cDCGAN)**: Generator receives `z` + **class label** → generates images of a *specific target class*

**Architecture change:**
- `Generator`:  `concat(z, class_embedding)` as input
- `Discriminator`: class label projected as an extra image channel (concatenated with radar image)

**Dataset Classes:**
| Label | Class |
|---|---|
| 0 | Pedestrian |
| 1 | Electric Scooter |
| 2 | Car |
| 3 | Background / Other |


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from numpy import expand_dims
from torchinfo import summary

# Class label map — update if your labels differ
CLASS_NAMES = {0: "Pedestrian", 1: "Electric Scooter", 2: "Car", 3: "Background"}
print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())


In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
DATASET_PATH = r"C:\\Users\\KRISHNA\\Desktop\\ALL GAN FINAL\\Dataset_19200_train_4800_test\\Dataset_19200_train_4800_test.npz"

IMAGE_SIZE   = (64, 64)
CHANNELS     = 1
BATCH_SIZE   = 100
LATENT_DIM   = 100
EMBED_DIM    = 50          # class embedding dimension (new hyperparameter)
NUM_CLASSES  = 4           # 0=Pedestrian, 1=E-Scooter, 2=Car, 3=Background
EPOCHS       = 30
LR           = 0.0002
BETA1        = 0.5
BETA2        = 0.999
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 📦 Data Loading

In [ ]:
class RadarDataset(Dataset):
    """Radar dataset with class labels preserved for conditioning."""

    def __init__(self, data: np.ndarray, labels: np.ndarray, image_size: tuple):
        H, W       = image_size
        orig_H, orig_W = data.shape[1], data.shape[2]
        x0, y0     = (orig_H - H) // 2, (orig_W - W) // 2

        cropped    = data[:, x0:x0+H, y0:y0+W].astype(np.float32)
        cropped    = (cropped / 255.0) * 2.0 - 1.0      # [-1, 1]
        self.images = torch.from_numpy(cropped[:, np.newaxis, :, :])
        self.labels = torch.from_numpy(labels.astype(np.int64))

    def __len__(self):          return len(self.labels)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]


archive       = np.load(DATASET_PATH, allow_pickle=False)
train_dataset = RadarDataset(archive["X_train"], archive["y_train"], IMAGE_SIZE)
test_dataset  = RadarDataset(archive["X_test"],  archive["y_test"],  IMAGE_SIZE)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

label_counts  = {k: (archive["y_train"] == k).sum() for k in range(NUM_CLASSES)}
print(f"Train: {len(train_dataset)}  |  Test: {len(test_dataset)}")
print("Class distribution in training set:")
for k, v in label_counts.items():
    print(f"  Class {k} ({CLASS_NAMES[k]:>16s}): {v:>5} samples")


## 🧠 Conditional Generator

**Key change vs. vanilla DCGAN:**
The input is no longer just `z` (100,) but `concat(z, class_embedding)` = (150,).
The class embedding is learned — the network learns a 50-dimensional representation for each class.


In [ ]:
class ConditionalGenerator(nn.Module):
    """
    cDCGAN Generator.
    Input  : noise z (B, latent_dim)  +  class label (B,)
    Output : synthetic radar image (B, 1, 64, 64)
    """

    def __init__(self, latent_dim, num_classes, embed_dim, channels, image_size):
        super().__init__()
        H, W = image_size
        self.init_h, self.init_w = H // 4, W // 4    # 16 × 16

        # ── Class label → dense embedding ────────────────────────────────
        self.label_emb = nn.Embedding(num_classes, embed_dim)

        # ── Generator trunk (input = latent_dim + embed_dim = 150) ────────
        in_features = latent_dim + embed_dim
        self.model  = nn.Sequential(
            nn.Linear(in_features, 128 * self.init_h * self.init_w),
            nn.ReLU(inplace=True),
            nn.Unflatten(1, (128, self.init_h, self.init_w)),

            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 128, 3, 1, 1),
            nn.BatchNorm2d(128, momentum=0.8),
            nn.ReLU(inplace=True),

            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, 3, 1, 1),
            nn.BatchNorm2d(64, momentum=0.8),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, channels, 3, 1, 1),
            nn.Tanh(),
        )

    def forward(self, z, labels):
        """
        Args:
            z      : noise tensor       (B, latent_dim)
            labels : class label tensor (B,)  — integer class ids
        Returns:
            fake images (B, 1, 64, 64)
        """
        emb   = self.label_emb(labels)      # (B, embed_dim)
        x     = torch.cat([z, emb], dim=1)  # (B, latent_dim + embed_dim)
        return self.model(x)


generator = ConditionalGenerator(LATENT_DIM, NUM_CLASSES, EMBED_DIM, CHANNELS, IMAGE_SIZE).to(DEVICE)
print(summary(generator, input_data=[torch.randn(1, LATENT_DIM).to(DEVICE),
                                     torch.zeros(1, dtype=torch.long).to(DEVICE)]))


## 🔍 Conditional Discriminator

**Key change vs. vanilla DCGAN:**
The class label is projected into a **spatial map** of shape `(1, 64, 64)` and
concatenated with the input image → in_channels becomes **2** instead of 1.
This forces the Discriminator to verify: *"Is this image both realistic AND the correct class?"*


In [ ]:
class ConditionalDiscriminator(nn.Module):
    """
    cDCGAN Discriminator.
    Input  : image (B, 1, 64, 64)  +  class label (B,)
    Output : real/fake probability (B, 1)
    """

    def __init__(self, channels, num_classes, image_size):
        super().__init__()
        H, W = image_size

        # ── Project class label → spatial map (1, H, W) ──────────────────
        self.label_emb = nn.Embedding(num_classes, H * W)
        self.H, self.W = H, W

        # ── Discriminator trunk (in_channels = image channels + 1) ────────
        in_ch = channels + 1   # 1 (image) + 1 (label map) = 2

        self.features = nn.Sequential(
            nn.Conv2d(in_ch, 32,  3, 2, 1), nn.LeakyReLU(0.2, True), nn.Dropout(0.25),
            nn.Conv2d(32,   64,  3, 2, 1), nn.BatchNorm2d(64),  nn.LeakyReLU(0.2, True), nn.Dropout(0.25),
            nn.Conv2d(64,   128, 3, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True), nn.Dropout(0.25),
            nn.Conv2d(128,  256, 3, 1, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True), nn.Dropout(0.25),
        )
        flat_dim = (H // 8) * (W // 8) * 256  # 8×8×256 = 16384

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, img, labels):
        """
        Args:
            img    : image tensor        (B, 1, 64, 64)
            labels : class label tensor  (B,)
        Returns:
            real/fake probability (B, 1)
        """
        emb  = self.label_emb(labels)                    # (B, H*W)
        emb  = emb.view(-1, 1, self.H, self.W)           # (B, 1, H, W)
        x    = torch.cat([img, emb], dim=1)              # (B, 2, H, W)
        return self.classifier(self.features(x))


discriminator = ConditionalDiscriminator(CHANNELS, NUM_CLASSES, IMAGE_SIZE).to(DEVICE)
print(summary(discriminator, input_data=[torch.randn(1, 1, 64, 64).to(DEVICE),
                                          torch.zeros(1, dtype=torch.long).to(DEVICE)]))


## 🏋️ Training the Conditional DCGAN

**What changes in the training loop vs. vanilla DCGAN:**
1. Real labels come from the dataset (not discarded like before)
2. Fake labels are **randomly sampled** from all classes
3. Both G and D receive class labels at every step


In [ ]:
import os, time

os.makedirs("cdcgan_outputs/checkpoints",    exist_ok=True)
os.makedirs("cdcgan_outputs/images",         exist_ok=True)

criterion   = nn.BCELoss()
optimizer_G = torch.optim.Adam(generator.parameters(),     lr=LR, betas=(BETA1, BETA2))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=LR, betas=(BETA1, BETA2))

d_losses, g_losses = [], []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    epoch_d, epoch_g = 0.0, 0.0

    for real_imgs, real_labels in train_loader:
        batch       = real_imgs.size(0)
        real_imgs   = real_imgs.to(DEVICE)
        real_labels = real_labels.to(DEVICE)

        valid = torch.ones(batch,  1, device=DEVICE)
        fake  = torch.zeros(batch, 1, device=DEVICE)

        # ── Step 1: Train Discriminator FIRST ─────────────────────────────
        optimizer_D.zero_grad()
        # Sample random class labels for fake images
        fake_labels = torch.randint(0, NUM_CLASSES, (batch,), device=DEVICE)
        z           = torch.randn(batch, LATENT_DIM, device=DEVICE)
        fake_imgs   = generator(z, fake_labels).detach()

        real_loss   = criterion(discriminator(real_imgs, real_labels), valid)
        fake_loss   = criterion(discriminator(fake_imgs, fake_labels), fake)
        d_loss      = (real_loss + fake_loss) / 2.0
        d_loss.backward()
        optimizer_D.step()

        # ── Step 2: Train Generator SECOND ────────────────────────────────
        optimizer_G.zero_grad()
        fake_labels = torch.randint(0, NUM_CLASSES, (batch,), device=DEVICE)
        z           = torch.randn(batch, LATENT_DIM, device=DEVICE)
        fake_imgs   = generator(z, fake_labels)
        g_loss      = criterion(discriminator(fake_imgs, fake_labels), valid)
        g_loss.backward()
        optimizer_G.step()

        epoch_d += d_loss.item()
        epoch_g += g_loss.item()

    avg_d = epoch_d / len(train_loader)
    avg_g = epoch_g / len(train_loader)
    d_losses.append(avg_d)
    g_losses.append(avg_g)
    elapsed = time.time() - t0

    print(f"Epoch [{epoch:02d}/{EPOCHS}]  D_Loss: {avg_d:.5f}  G_Loss: {avg_g:.5f}  "
          f"({elapsed:.1f}s)")

    # Save checkpoint every 5 epochs
    if epoch % 5 == 0:
        torch.save({
            "epoch": epoch,
            "generator_state_dict":     generator.state_dict(),
            "discriminator_state_dict": discriminator.state_dict(),
        }, f"cdcgan_outputs/checkpoints/cdcgan_epoch_{epoch:04d}.pt")
        print(f"  ✅ Checkpoint saved.")

print("\n🎉 Training Complete!")


## 📊 Training Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, EPOCHS+1), d_losses, label="Discriminator Loss", color="#E74C3C", lw=2)
ax.plot(range(1, EPOCHS+1), g_losses, label="Generator Loss",     color="#2E86C1", lw=2)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss",  fontsize=12)
ax.set_title("Conditional DCGAN — Training Loss (Radar View Generator)", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("cdcgan_outputs/loss_curves.png", dpi=120)
plt.show()
print("Loss curves saved.")


## 🎨 Generate Images — Class by Class

This is the **key advantage of cDCGAN** over vanilla DCGAN.
We can now ask: *"Give me 8 images of a Car"* or *"Give me 8 images of a Pedestrian"*
by simply fixing the class label passed to the Generator.


In [ ]:
generator.eval()

N_PER_CLASS = 8
fig, axes   = plt.subplots(NUM_CLASSES, N_PER_CLASS,
                            figsize=(N_PER_CLASS * 1.8, NUM_CLASSES * 2.0))

with torch.inference_mode():
    for cls_id in range(NUM_CLASSES):
        torch.manual_seed(42)                          # reproducible grid
        z      = torch.randn(N_PER_CLASS, LATENT_DIM, device=DEVICE)
        labels = torch.full((N_PER_CLASS,), cls_id, dtype=torch.long, device=DEVICE)
        imgs   = generator(z, labels).squeeze(1).cpu().numpy()
        imgs   = np.clip((imgs + 1.0) / 2.0, 0, 1)   # [-1,1] → [0,1]

        for col, img in enumerate(imgs):
            ax = axes[cls_id, col]
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(f"Class {cls_id}\n{CLASS_NAMES[cls_id]}",
                              fontsize=9, rotation=0,
                              labelpad=70, va="center")

fig.suptitle("Conditional DCGAN — Generated Radar Images per Class", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("cdcgan_outputs/images/generated_per_class.png", dpi=120, bbox_inches="tight")
plt.show()
print("Per-class grid saved.")

generator.train()


## 🔍 Generate a 5×5 Grid for a Specific Class

In [ ]:
def generate_class_grid(target_class: int, n: int = 25, seed: int = 42):
    """Generate n images of a single specified class."""
    generator.eval()
    side = int(n ** 0.5)

    with torch.inference_mode():
        torch.manual_seed(seed)
        z      = torch.randn(n, LATENT_DIM, device=DEVICE)
        labels = torch.full((n,), target_class, dtype=torch.long, device=DEVICE)
        imgs   = generator(z, labels).squeeze(1).cpu().numpy()
        imgs   = np.clip((imgs + 1.0) / 2.0, 0, 1)

    fig, axes = plt.subplots(side, side, figsize=(side * 1.8, side * 1.8))
    for ax, img in zip(axes.flat, imgs):
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
    name = CLASS_NAMES[target_class]
    fig.suptitle(f"Generated: {name}  (Class {target_class}) — 25 samples",
                  fontsize=12)
    plt.tight_layout()
    path = f"cdcgan_outputs/images/class_{target_class}_{name.replace(' ','_')}.png"
    plt.savefig(path, dpi=120)
    plt.show()
    print(f"Saved → {path}")
    generator.train()

# ── Try each class ─────────────────────────────────────────────────────────────
for cls in range(NUM_CLASSES):
    generate_class_grid(target_class=cls, n=25, seed=42)


## 🆚 Side-by-Side Class Comparison
The most impressive demo: **same noise vector `z`, different class labels → different images.**
This proves the Generator has learned class-specific visual features.


In [ ]:
generator.eval()

N_SAMPLES = 6
torch.manual_seed(99)
z_fixed   = torch.randn(N_SAMPLES, LATENT_DIM, device=DEVICE)  # SAME z for all classes

fig, axes = plt.subplots(NUM_CLASSES, N_SAMPLES,
                          figsize=(N_SAMPLES * 2.0, NUM_CLASSES * 2.2))

with torch.inference_mode():
    for row, cls_id in enumerate(range(NUM_CLASSES)):
        labels = torch.full((N_SAMPLES,), cls_id, dtype=torch.long, device=DEVICE)
        imgs   = generator(z_fixed, labels).squeeze(1).cpu().numpy()
        imgs   = np.clip((imgs + 1.0) / 2.0, 0, 1)
        for col, img in enumerate(imgs):
            ax = axes[row, col]
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(f"{CLASS_NAMES[cls_id]}",
                              fontsize=10, rotation=0,
                              labelpad=90, va="center",
                              fontweight="bold")

fig.suptitle("Same Noise z → Different Class Labels → Different Radar Images\n"
             "(Conditional DCGAN — Class Control Demonstration)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("cdcgan_outputs/images/class_comparison_same_z.png",
            dpi=120, bbox_inches="tight")
plt.show()
print("Comparison grid saved.")
generator.train()


## 💾 Save Final Model

In [ ]:
torch.save({
    "epoch":                    EPOCHS,
    "latent_dim":               LATENT_DIM,
    "num_classes":              NUM_CLASSES,
    "embed_dim":                EMBED_DIM,
    "class_names":              CLASS_NAMES,
    "generator_state_dict":     generator.state_dict(),
    "discriminator_state_dict": discriminator.state_dict(),
    "d_losses":                 d_losses,
    "g_losses":                 g_losses,
}, "cdcgan_outputs/checkpoints/cdcgan_final.pt")

print("✅ Final model saved → cdcgan_outputs/checkpoints/cdcgan_final.pt")
print(f"   Generator     params : {sum(p.numel() for p in generator.parameters()):,}")
print(f"   Discriminator params : {sum(p.numel() for p in discriminator.parameters()):,}")


## 📋 Vanilla DCGAN vs. Conditional DCGAN — Summary

| Aspect | Vanilla DCGAN | Conditional DCGAN |
|---|---|---|
| Generator input | `z` (noise only) | `concat(z, class_embedding)` |
| Discriminator input | image only | image + class label map |
| Output control | Random class | **Specific class on demand** |
| New hyperparameter | — | `embed_dim = 50` |
| Label use in training | Labels discarded | Labels used for conditioning |
| Use case | General synthesis | **Targeted data augmentation** |

### ✅ Practical Benefits
- Generate **balanced datasets** — create more samples for under-represented classes
- **Targeted augmentation** — augment only the "Pedestrian" class if that class is rare
- **Ablation study** — compare what the model learned for each class independently
- Easier to **evaluate quality per class** instead of overall
